In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import time, json

print("CUDA available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

INTERIM_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim"
SENTIMENT_MODEL_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim/interim/distilbert_sentiment"

tokenizer = DistilBertTokenizerFast.from_pretrained(SENTIMENT_MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(SENTIMENT_MODEL_PATH)
model.to(device)
model.eval()

print("model loaded, device:", next(model.parameters()).device)

CUDA available: True
using device: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

model loaded, device: cuda:0


In [2]:
# CELL 2 — Load and clean RT reviews

def clean_rt_reviews(rt_reviews, text_col="review_content"):
    before = len(rt_reviews)
    cleaned = rt_reviews.dropna(subset=[text_col]).copy()
    cleaned = cleaned[cleaned[text_col].str.len() > 0]
    print(f"dropped {before - len(cleaned)} empty/null reviews ({100*(before-len(cleaned))/before:.1f}%)")
    return cleaned

rt_reviews = pd.read_parquet(f"{INTERIM_PATH}/rt_reviews_matched.parquet")
print("rt_reviews shape:", rt_reviews.shape)

rt_clean = clean_rt_reviews(rt_reviews)
print("rt_clean shape:", rt_clean.shape)
print(rt_clean[["movieId", "review_content"]].head())

rt_reviews shape: (828444, 9)
dropped 44966 empty/null reviews (5.4%)
rt_clean shape: (783478, 9)
   movieId                                     review_content
0    74530  A fantasy adventure that fuses Greek mythology...
1    74530  Uma Thurman as Medusa, the gorgon with a coiff...
2    74530  With a top-notch cast and dazzling special eff...
3    74530  Whether audiences will get behind The Lightnin...
4    74530  What's really lacking in The Lightning Thief i...


In [3]:
# CELL 3 — Score every RT review with the trained classifier

def score_all_reviews(rt_clean, model, tokenizer, device="cpu",
                       text_col="review_content", batch_size=64):
    out = rt_clean.copy()
    texts = out[text_col].tolist()

    all_probs = []
    start = time.time()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        encoding = tokenizer(
            batch_texts, truncation=True, padding=True,
            max_length=256, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**encoding).logits
            probs = torch.softmax(logits, dim=1)[:, 1]  # probability of "positive"
        all_probs.extend(probs.cpu().tolist())

        if (i // batch_size) % 500 == 0:
            elapsed = time.time() - start
            pct = 100 * (i + len(batch_texts)) / len(texts)
            print(f"  scored {i + len(batch_texts)}/{len(texts)} ({pct:.1f}%) — {elapsed:.1f}s elapsed")

    out["sentiment_prob"] = all_probs
    out["sentiment_label"] = (out["sentiment_prob"] >= 0.5).astype(int)
    return out


start = time.time()
scored = score_all_reviews(rt_clean, model, tokenizer, device=device, batch_size=64)
elapsed = time.time() - start

print(f"\ntotal scoring time: {elapsed:.1f}s")
print("scored shape:", scored.shape)
print(scored[["movieId", "review_content", "sentiment_prob", "sentiment_label"]].head(10))

  scored 64/783478 (0.0%) — 0.5s elapsed
  scored 32064/783478 (4.1%) — 52.7s elapsed
  scored 64064/783478 (8.2%) — 118.3s elapsed
  scored 96064/783478 (12.3%) — 182.3s elapsed
  scored 128064/783478 (16.3%) — 247.1s elapsed
  scored 160064/783478 (20.4%) — 311.0s elapsed
  scored 192064/783478 (24.5%) — 374.9s elapsed
  scored 224064/783478 (28.6%) — 439.6s elapsed
  scored 256064/783478 (32.7%) — 503.4s elapsed
  scored 288064/783478 (36.8%) — 569.1s elapsed
  scored 320064/783478 (40.9%) — 633.6s elapsed
  scored 352064/783478 (44.9%) — 699.3s elapsed
  scored 384064/783478 (49.0%) — 764.0s elapsed
  scored 416064/783478 (53.1%) — 829.2s elapsed
  scored 448064/783478 (57.2%) — 894.8s elapsed
  scored 480064/783478 (61.3%) — 958.7s elapsed
  scored 512064/783478 (65.4%) — 1020.9s elapsed
  scored 544064/783478 (69.4%) — 1083.3s elapsed
  scored 576064/783478 (73.5%) — 1146.3s elapsed
  scored 608064/783478 (77.6%) — 1208.4s elapsed
  scored 640064/783478 (81.7%) — 1270.1s elapsed


In [4]:
# CELL 4 — Aggregate to per-movie sentiment, save results

def aggregate_movie_sentiment(scored_reviews, movie_col="movieId"):
    agg = scored_reviews.groupby(movie_col).agg(
        sentiment_score=("sentiment_prob", "mean"),
        review_count=("sentiment_prob", "count"),
        pct_positive=("sentiment_label", "mean"),
    ).reset_index()
    return agg


movie_sentiment = aggregate_movie_sentiment(scored)
print("movie_sentiment shape:", movie_sentiment.shape)
print("\ndistribution:")
print(movie_sentiment.describe())

print("\nsample:")
print(movie_sentiment.head(10))

# Quick sanity check: highest and lowest sentiment movies (with enough reviews to trust)
reliable = movie_sentiment[movie_sentiment["review_count"] >= 20]
print("\ntop 5 highest sentiment (>=20 reviews):")
print(reliable.sort_values("sentiment_score", ascending=False).head())
print("\ntop 5 lowest sentiment (>=20 reviews):")
print(reliable.sort_values("sentiment_score", ascending=True).head())

# Save everything
scored.to_parquet("/kaggle/working/rt_reviews_scored.parquet", index=False)
movie_sentiment.to_parquet("/kaggle/working/movie_sentiment.parquet", index=False)

results_summary = {
    "total_reviews_scored": len(scored),
    "movies_with_sentiment": len(movie_sentiment),
    "spot_check_accuracy_vs_review_type": 0.8240,
    "imdb_val_accuracy": 0.9247,
    "review_count_stats": {
        "min": int(movie_sentiment["review_count"].min()),
        "median": float(movie_sentiment["review_count"].median()),
        "max": int(movie_sentiment["review_count"].max()),
    },
    "sentiment_score_stats": {
        "mean": float(movie_sentiment["sentiment_score"].mean()),
        "std": float(movie_sentiment["sentiment_score"].std()),
    },
}
with open("/kaggle/working/sentiment_scoring_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("\nsaved: rt_reviews_scored.parquet, movie_sentiment.parquet, sentiment_scoring_results.json")
print(json.dumps(results_summary, indent=2))

movie_sentiment shape: (10950, 4)

distribution:
             movieId  sentiment_score  review_count  pct_positive
count   10950.000000     10950.000000  10950.000000  10950.000000
mean    78363.366393         0.509842     71.550502      0.491926
std     68861.371056         0.203604     91.220109      0.238952
min         1.000000         0.002305      1.000000      0.000000
25%      5701.500000         0.349647     12.000000      0.300000
50%     70761.000000         0.522301     33.000000      0.500000
75%    137151.500000         0.674656    101.000000      0.680000
max    208939.000000         0.996763    948.000000      1.000000

sample:
   movieId  sentiment_score  review_count  pct_positive
0        1         0.803544            85      0.811765
1        2         0.702697            17      0.647059
2        3         0.339732             7      0.285714
3        4         0.429719            21      0.380952
4        5         0.420973            10      0.500000
5        6  

In [5]:
# Quick check: how many movies now have very few reviews after the empty-text drop?
low_count = movie_sentiment[movie_sentiment["review_count"] < 4]
print(f"movies with < 4 reviews (was the old Week 1 floor): {len(low_count)}")
print(low_count.sort_values("review_count").head(10))

movies with < 4 reviews (was the old Week 1 floor): 283
       movieId  sentiment_score  review_count  pct_positive
48          68         0.047814             1           0.0
81         131         0.948801             1           1.0
1049      2199         0.394225             1           0.0
10013   185377         0.002663             1           0.0
1865      3876         0.415168             1           0.0
7526    120386         0.011597             1           0.0
7681    123489         0.098813             1           0.0
2568      5344         0.002305             1           0.0
2433      5018         0.171218             1           0.0
2669      5547         0.500294             1           1.0


In [6]:
# Map top/bottom sentiment movies to titles for a real sanity check
movies_master = pd.read_parquet(f"{INTERIM_PATH}/movies_master.parquet")
movieid_to_title = dict(zip(movies_master["movieId"], movies_master["title"]))

reliable = movie_sentiment[movie_sentiment["review_count"] >= 20].copy()
reliable["title"] = reliable["movieId"].map(movieid_to_title)

print("top 5 highest sentiment (>=20 reviews):")
print(reliable.sort_values("sentiment_score", ascending=False)[["title", "sentiment_score", "review_count"]].head())

print("\ntop 5 lowest sentiment (>=20 reviews):")
print(reliable.sort_values("sentiment_score", ascending=True)[["title", "sentiment_score", "review_count"]].head())

top 5 highest sentiment (>=20 reviews):
                                  title  sentiment_score  review_count
2576               Rambling Rose (1991)         0.985403            24
3554  Ugetsu (Ugetsu monogatari) (1953)         0.949353            26
5984       Riot in Cell Block 11 (1954)         0.941429            26
1010               Atlantic City (1980)         0.929000            23
6636             Planet of Snail (2011)         0.921267            28

top 5 lowest sentiment (>=20 reviews):
                         title  sentiment_score  review_count
2630        Wagons East (1994)         0.071508            23
4835  Who's Your Caddy? (2007)         0.075054            34
7310            Reclaim (2014)         0.075349            20
8805          Get a Job (2016)         0.077291            22
4182      King's Ransom (2005)         0.082953            47


In [7]:
# CELL 5 — Add a reliability threshold, finalize and re-save

MIN_RELIABLE_REVIEWS = 5  # movies below this get flagged as "low confidence" for the re-ranker

movie_sentiment["reliable"] = movie_sentiment["review_count"] >= MIN_RELIABLE_REVIEWS

print(f"movies with reliable sentiment (>= {MIN_RELIABLE_REVIEWS} reviews): "
      f"{movie_sentiment['reliable'].sum()} ({100*movie_sentiment['reliable'].mean():.1f}%)")
print(f"movies below reliability threshold: {(~movie_sentiment['reliable']).sum()}")

movie_sentiment.to_parquet("/kaggle/working/movie_sentiment.parquet", index=False)
print("\nre-saved movie_sentiment.parquet with 'reliable' flag")
print(movie_sentiment.head())

movies with reliable sentiment (>= 5 reviews): 10444 (95.4%)
movies below reliability threshold: 506

re-saved movie_sentiment.parquet with 'reliable' flag
   movieId  sentiment_score  review_count  pct_positive  reliable
0        1         0.803544            85      0.811765      True
1        2         0.702697            17      0.647059      True
2        3         0.339732             7      0.285714      True
3        4         0.429719            21      0.380952      True
4        5         0.420973            10      0.500000      True
